# **Seq2Seq 실습**
`nn.Embedding`과 `nn.LSTM`을 이용하여 간단한 영어 → 한국어 번역 모델을 만들어보자.


간단한 문장 패턴을 이용해 영어 문장을 한국어로 번역한다.

`I went to school yesterday` → `나는 어제 학교에 갔다`

영어와 한국어는 **단어 순서가 다르기 때문에** Encoder-Decoder 구조를 보기 좋은 예시이다.


In [11]:
import torch
import torch.nn as nn

torch.manual_seed(1)

BOS = 1 # Beginning Of Sentence
EOS = 2 # End Of Sentence

## **1. 학습 데이터 만들기**

아래 단어들을 조합하여 여러 개의 영어-한국어 문장 쌍을 만든다.

일부 문장은 학습에서 제외하고 마지막에 번역 결과를 확인한다.

- 아래 코드를 실행하고 이해해보자.


In [12]:
subjects = {
    "i": "나는",
    "you": "너는",
    "he": "그는",
    "she": "그녀는"
}

places = {
    "school": "학교에",
    "library": "도서관에",
    "park": "공원에",
    "museum": "박물관에"
}

times = {
    "yesterday": "어제",
    "today": "오늘",
    "morning": "아침에"
}

test_cases = {
    ("i", "school", "yesterday"),
    ("you", "library", "today"),
    ("he", "park", "morning"),
    ("she", "museum", "yesterday")
}

train_pairs = []
test_pairs = []

# 결합해서 문장 만들기
for eng_sub, kor_sub in subjects.items():
    for eng_place, kor_place in places.items():
        for eng_time, kor_time in times.items():

            eng = f"{eng_sub} went to {eng_place} {eng_time}"
            kor = f"{kor_sub} {kor_time} {kor_place} 갔다"

            if (eng_sub, eng_place, eng_time) in test_cases:
                test_pairs.append((eng, kor))
            else:
                train_pairs.append((eng, kor))

In [13]:
print("학습 데이터 :", len(train_pairs))
print("테스트 데이터 :", len(test_pairs))
print(train_pairs[0])

학습 데이터 : 44
테스트 데이터 : 4
('i went to school today', '나는 오늘 학교에 갔다')


## **2. Tokenization**

`nn.Embedding`에 넣기 위해 영어 단어와 한국어 단어를 각각 정수 index로 변환한다.

**train_pairs와 test_pairs를 더해 all_pairs로 만들어보자.**

In [14]:
# code here
all_pairs = train_pairs + test_pairs
print(len(all_pairs))
print(all_pairs)

48
[('i went to school today', '나는 오늘 학교에 갔다'), ('i went to school morning', '나는 아침에 학교에 갔다'), ('i went to library yesterday', '나는 어제 도서관에 갔다'), ('i went to library today', '나는 오늘 도서관에 갔다'), ('i went to library morning', '나는 아침에 도서관에 갔다'), ('i went to park yesterday', '나는 어제 공원에 갔다'), ('i went to park today', '나는 오늘 공원에 갔다'), ('i went to park morning', '나는 아침에 공원에 갔다'), ('i went to museum yesterday', '나는 어제 박물관에 갔다'), ('i went to museum today', '나는 오늘 박물관에 갔다'), ('i went to museum morning', '나는 아침에 박물관에 갔다'), ('you went to school yesterday', '너는 어제 학교에 갔다'), ('you went to school today', '너는 오늘 학교에 갔다'), ('you went to school morning', '너는 아침에 학교에 갔다'), ('you went to library yesterday', '너는 어제 도서관에 갔다'), ('you went to library morning', '너는 아침에 도서관에 갔다'), ('you went to park yesterday', '너는 어제 공원에 갔다'), ('you went to park today', '너는 오늘 공원에 갔다'), ('you went to park morning', '너는 아침에 공원에 갔다'), ('you went to museum yesterday', '너는 어제 박물관에 갔다'), ('you went to museum today', '너는 오늘 박물관에 갔다'), 

**all_pairs에 존재하는 모든 영단어들을 찾고, eng2idx라는 딕셔너리에 각 단어들에 대해 숫자를 부여하자.**

숫자는 3부터 시작하게 하도록 한다.

In [15]:
# code here
# 1. tokenizing 부터 해결
eng_words = [word.split() for word, _ in all_pairs]


In [16]:
eng2idx = {}
for sentence in eng_words:
    for word in sentence:
        if eng2idx.get(word) == None:
            # BOS = 1, EOS = 2
            eng2idx[word] = len(eng2idx) + 3 

In [17]:
print(eng2idx)
print(len(eng2idx))

{'i': 3, 'went': 4, 'to': 5, 'school': 6, 'today': 7, 'morning': 8, 'library': 9, 'yesterday': 10, 'park': 11, 'museum': 12, 'you': 13, 'he': 14, 'she': 15}
13


**all_pairs에 존재하는 모든 한글 단어들을 찾고, kor2idx라는 딕셔너리에 각 단어들에 대해 숫자를 부여하자.**

숫자는 3부터 시작하게 하도록 한다.

In [ ]:
# code here
kor_words = [i.split() for _, i in all_pairs]

kor2idx = {}
for sentence in kor_words:
    for word in sentence:
        if kor2idx.get(word) == None:
            kor2idx[word] = len(kor2idx) + 3

**kor2idx의 key, value pair를 뒤집어 숫자 : 단어의 형태로 새로운 idx2kor이라는 딕셔너리를 만들자.**

In [ ]:
# code here
idx2kor = {val : key for key, val in kor2idx.items()}

{3: '나는', 4: '오늘', 5: '학교에', 6: '갔다', 7: '아침에', 8: '어제', 9: '도서관에', 10: '공원에', 11: '박물관에', 12: '너는', 13: '그는', 14: '그녀는'}


**vocab 크기를 설정한다.**
- eng_vocab_size : eng2idx의 크기에 3을 더한 크기로 설정한다.
- kor_vocab_size : kor2idx의 크기에 3을 더한 크기로 설정한다.

In [25]:
# code here
eng_vocab_size = len(eng2idx) + 3
kor_vocab_size = len(kor2idx) + 3

**데이터를텐서화시킨다.**

- X : train_pairs의 모든 영어로 된 문장을 공백으로 분리하여 eng2idx에 토큰화된 숫자로 변환한 데이터를 생성한다.

- y : train_pairs의 모든 한글로 된 문장을 공백으로 분리하여 kor2idx에 토큰화된 숫자로 변환한 데이터를 생성한다.
- 이 데이터의 끝에는 EOS를 추가한다.

In [ ]:
eng = [[eng2idx[j] for j in i[0].split()] for i in train_pairs]

[[3, 4, 5, 6, 7], [3, 4, 5, 6, 8], [3, 4, 5, 9, 10], [3, 4, 5, 9, 7], [3, 4, 5, 9, 8], [3, 4, 5, 11, 10], [3, 4, 5, 11, 7], [3, 4, 5, 11, 8], [3, 4, 5, 12, 10], [3, 4, 5, 12, 7], [3, 4, 5, 12, 8], [13, 4, 5, 6, 10], [13, 4, 5, 6, 7], [13, 4, 5, 6, 8], [13, 4, 5, 9, 10], [13, 4, 5, 9, 8], [13, 4, 5, 11, 10], [13, 4, 5, 11, 7], [13, 4, 5, 11, 8], [13, 4, 5, 12, 10], [13, 4, 5, 12, 7], [13, 4, 5, 12, 8], [14, 4, 5, 6, 10], [14, 4, 5, 6, 7], [14, 4, 5, 6, 8], [14, 4, 5, 9, 10], [14, 4, 5, 9, 7], [14, 4, 5, 9, 8], [14, 4, 5, 11, 10], [14, 4, 5, 11, 7], [14, 4, 5, 12, 10], [14, 4, 5, 12, 7], [14, 4, 5, 12, 8], [15, 4, 5, 6, 10], [15, 4, 5, 6, 7], [15, 4, 5, 6, 8], [15, 4, 5, 9, 10], [15, 4, 5, 9, 7], [15, 4, 5, 9, 8], [15, 4, 5, 11, 10], [15, 4, 5, 11, 7], [15, 4, 5, 11, 8], [15, 4, 5, 12, 7], [15, 4, 5, 12, 8]]


In [35]:
# code here
X = torch.tensor(
    [[eng2idx[j] for j in i[0].split()] for i in train_pairs]
)

In [38]:
y = torch.tensor(
    [[kor2idx[j] for j in i[1].split()] + [EOS] for i in train_pairs]
)

In [39]:
print("X :", X.shape)
print("y :", y.shape)

X : torch.Size([44, 5])
y : torch.Size([44, 5])


In [40]:
print("영어 :", train_pairs[0][0], X[0])
print("한국어 :", train_pairs[0][1], y[0])

영어 : i went to school today tensor([3, 4, 5, 6, 7])
한국어 : 나는 오늘 학교에 갔다 tensor([3, 4, 5, 6, 2])


## **3. Encoder 생성하기**

`영어 문장 → Embedding → LSTM → Context State`

Encoder는 영어 문장 전체를 읽고 마지막 `hidden`, `cell`을 Decoder에 전달한다.


In [41]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        self.lstm = nn.LSTM(
            embedding_size,
            hidden_size,
            batch_first=True
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, cell) = self.lstm(x)
        return hidden, cell

## **4. Decoder 생성하기**

`이전 한국어 Token → Embedding → LSTM → Linear → 다음 한국어 Token 예측`
- Decoder는 한 번에 한국어 Token 하나를 예측한다.


In [42]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        self.lstm = nn.LSTM(
            embedding_size,
            hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(self, x, hidden, cell):
      x = self.embedding(x).reshape(-1, 1, self.embedding.embedding_dim)
      output, (hidden, cell) = self.lstm(x, (hidden, cell))
      pred = self.fc(output.reshape(-1, output.size(-1)))
      return pred, hidden, cell

## **5. 모델 생성하기**

- embedding size 16
- hidden size : 64

In [ ]:
# code here

**encoder 인스턴스를 생성한다.**
- eng_vocab_size와 embedding_size, hidden_size를 parameter로 받아준다.

In [53]:
# code here
encoder = Encoder(
    vocab_size= eng_vocab_size,
    embedding_size= 16,
    hidden_size= 64
)

**decoder 인스턴스를 생성한다.**
- kor_vocab_size와 embedding_size, hidden_size를 parameter로 받아준다.

In [45]:
# code here
decoder = Decoder(
    vocab_size= kor_vocab_size,
    embedding_size=16,
    hidden_size=64
)

**Loss 함수와 Optimizer를 설정한다.**
- CrosEntropyLoss()를 활용한다.
- Optimizer : Adam
- encoder와 decoder의 parameter를 모두 전달한다.
```
list(encoder.parameters()) + list(decoder.parameters())
```
- 학습률은 0.001로 설정한다.

In [50]:
# code here
criterion = nn.CrossEntropyLoss()

In [54]:
# encoder.parametere() + decoder.parameters()
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr = 0.001)

## **6. 모델 학습**

* Encoder가 영어 문장을 읽는다.
* Decoder의 첫 입력은 `<BOS>`이다.
* Teacher Forcing을 사용하여 한국어 문장을 한 Token씩 학습한다.
  - teacher-forcing-ratio (정답을 다음 시점의 입력으로 사용할 확률)을 50% (0.5)로 설정한다.
- epoch은 1000으로 설정한다.


In [ ]:
epochs = 1000
teacher_forcing_ratio = 0.5633

In [55]:
encoder.train()
decoder.train()

for epoch in range(epochs):
    optimizer.zero_grad()

    hidden, cell = encoder(X)

    decoder_input = torch.full(
        (len(X),),
        BOS,
        dtype=torch.long
    )

    loss = 0

    for t in range(y.size(1)):
        pred, hidden, cell = decoder(
            decoder_input,
            hidden,
            cell
        )

        loss += criterion(
            pred,
            y[:, t]
        )

        use_teacher = (
            torch.rand(1).item() < teacher_forcing_ratio
        )

        if use_teacher:
            decoder_input = y[:, t]
        else:
            decoder_input = pred.argmax(dim=1)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch + 1:3d} | "
            f"Loss {loss.item() / y.size(1):.4f}"
        )

Epoch 100 | Loss 0.7109
Epoch 200 | Loss 0.0914
Epoch 300 | Loss 0.0289
Epoch 400 | Loss 0.0152
Epoch 500 | Loss 0.0097
Epoch 600 | Loss 0.0068
Epoch 700 | Loss 0.0051
Epoch 800 | Loss 0.0040
Epoch 900 | Loss 0.0032
Epoch 1000 | Loss 0.0026


## **7. 학습하지 않은 문장 번역하기**

Teacher Forcing 없이 모델이 직접 다음 Token을 예측한다.


In [56]:
encoder.eval()
decoder.eval()

with torch.no_grad():

    for eng, answer in test_pairs:

        x = torch.tensor([[
            eng2idx[word]
            for word in eng.split()
        ]])

        hidden, cell = encoder(x)

        decoder_input = torch.tensor([BOS])

        result = []

        for _ in range(6):
            pred, hidden, cell = decoder(
                decoder_input,
                hidden,
                cell
            )

            decoder_input = pred.argmax(dim=1)

            token = decoder_input.item()

            if token == EOS:
                break

            result.append(
                idx2kor[token]
            )

        print("영어 :", eng)
        print("정답 :", answer)
        print("예측 :", " ".join(result))
        print()

영어 : i went to school yesterday
정답 : 나는 어제 학교에 갔다
예측 : 나는 어제 학교에 갔다

영어 : you went to library today
정답 : 너는 오늘 도서관에 갔다
예측 : 너는 오늘 도서관에 갔다

영어 : he went to park morning
정답 : 그는 아침에 공원에 갔다
예측 : 그는 아침에 공원에 갔다

영어 : she went to museum yesterday
정답 : 그녀는 어제 박물관에 갔다
예측 : 그녀는 어제 박물관에 갔다

